# LLMOps Assignment 2 — API Cost, Latency and Prompt Version Management
**Course:** MLOps · **Student:** JALADI JEESHITHA SAI · **BITS ID:** 2025AE05875
**Scenario:** EduAssist AI - Academic question answering assistant that must return strict JSON.


# 1. Difference Between MLOps and LLMOps

| Aspect | MLOps | LLMOps |
|---|---|---|
| Main focus | Managing trained models and the data pipelines used to train them | Managing prompts, model versions, retrieved information and safety rules |
| What changes frequently | Data and models need to be updated or retrained when there are changes | Prompts and model providers can be changed frequently without retraining the model |
| Cost | Mainly depends on training resources, computing power and model deployment | Mainly depends on API usage, where input and output tokens are charged separately |
| Latency | Depends on model size, hardware and infrastructure | Depends on prompt length, context size, output length, model selection and API response time |
| Evaluation | Usually measured using metrics like accuracy and ROC-AUC on test data | Evaluated based on response quality, correctness, whether the answer is based on the given context, safety and human feedback |
| Versioning | Maintains versions of datasets and trained models | Maintains versions of prompts and models, including prompt changes, history and production versions |
| Common failures | Data changes or drift can reduce model accuracy over time | Changes in prompts or provider models can affect responses, and the model may also generate incorrect formats like invalid JSON |


In [ ]:
import json
import urllib.request

import pandas as pd
import mlflow

from eduassist import config
from eduassist.config import get_settings, load_questions
from eduassist.llm_client import LLMClient
from eduassist.workflow import setup_mlflow, task1_task2, task3, write_metadata

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

settings = get_settings()
print("Settings (safe to show - no secrets):")
for k, v in settings.public_dict().items():
    print(f"  {k:18s} {v}")
_ = settings.api_key()
print("api key loaded from environment (hidden)")

if settings.tracking_uri.startswith("http"):
    try:
        urllib.request.urlopen(settings.tracking_uri + "/health", timeout=5)
    except Exception as exc:
        raise RuntimeError("MLflow server not reachable. In a terminal run: "
                           "bash scripts/start_mlflow.sh") from exc
setup_mlflow(settings)
client = LLMClient(settings)
questions = load_questions()
config.OUTPUTS_DIR.mkdir(exist_ok=True)
print("\nMLflow", mlflow.__version__, "| tracking:", settings.tracking_uri,
      "| experiment:", config.EXPERIMENT_NAME)
print("API:", client.api_version)
pd.DataFrame(questions)[["id", "question"]]

## 1 - Calling an LLM and measuring latency and cost
Each of the five questions is sent once with prompt v1. For every request the client records question ID, prompt version, model name (as returned by the API), start/end time, latency and status. Failed calls are retried at most twice on rate limit/server errors, the number of attempts is recorded.

In [ ]:
t12 = task1_task2(client, questions, label="v1")
t12["latency_table"]

In [ ]:
stats = pd.DataFrame([t12["latency_stats"]]).rename(
    columns=lambda c: f"{c}_latency_s").round(4)
stats

In [ ]:
slow = t12["slowest"]
req = t12["requests"]
print(f"Slowest request: {slow['question_id']} - {slow['latency_seconds']:.3f} s "
      f"({slow['latency_seconds'] / t12['latency_stats']['mean']:.2f} x the mean)")
print(f"Output tokens {slow['output_tokens']} vs mean {req['output_tokens'].mean():.1f}; "
f"Input tokens {slow['input_tokens']} vs mean {req['input_tokens'].mean():.1f}; "
f"Attempts {slow['attempts']}")
print("First request of the session:", slow['question_id'] == req.iloc[0]['question_id'])
print("\nCorrelation latency ~ output tokens:",round(req['latency_seconds'].corr(req['output_tokens']), 3))

**Interpretation:**

## 2 - Token counting and cost
Token counts come from the `usage` field of every API response (`prompt_tokens`, `completion_tokens`).
Input cost = input tokens / 1,000,000 × input price per million  
Output cost = output tokens / 1,000,000 × output price per million  
Total cost per request = input cost + output cost  
Projected monthly cost = average cost per request × 10,000

In [ ]:
print(f"Model: {settings.model} | API: {client.api_version}")
print(f"Pricing: input {settings.price_input_per_m} / output {settings.price_output_per_m} "
      f"{settings.currency} per 1M tokens | pricing date {settings.price_date} "
      f"| source {settings.price_source}")
ct = t12["cost_table"].copy()
for c in ["input_cost", "output_cost", "total_cost"]:
    ct[c] = ct[c].map(lambda v: f"{v:.8f}")
ct

In [ ]:
avg = t12["avg_cost"]
monthly = t12["monthly_cost"]
print(f"Average cost per request : {avg:.8f} {settings.currency}")
print(f"Projected monthly cost   : {monthly:.4f} {settings.currency} for 10,000 requests")
r = t12["requests"]
in_share = r["input_cost"].sum() / max(r["total_cost"].sum(), 1e-12)
print(f"Share of cost from input tokens: {in_share:.0%}, output tokens: {1 - in_share:.0%}")

# Effect of shorter answers on the monthly bill
half_out = (r["input_cost"] + r["output_cost"] / 2).mean() * 10_000
print(f"What-if responses were 50% shorter      : {half_out:.4f} {settings.currency}/month")

**Cost-optimisation recommendation :**

## 3 - Prompt versioning and MLflow evaluation

1. registers v1, v2, v3 in the **MLflow Prompt Registry** under one prompt name with commit messages (re-running does not create duplicates);
2. runs each version on the same five questions in its own MLflow run (prompt loaded from the registry inside the run, which links prompt and run);
3. logs `avg_latency_seconds`, `avg_input_tokens`, `avg_output_tokens`, `avg_cost_per_request`, `valid_json` (+ schema validity, keyword-coverage quality score, projected monthly cost);
4. runs `mlflow.genai.evaluate` with boolean scorers `valid_json` and `schema_valid` — MLflow aggregates the per-response feedback into a mean = pass rate;
5. promotes the best version to the alias **`production`**.

`valid_json = valid JSON responses / total responses` (strict: the whole response must parse as a JSON object — code fences or extra text count as invalid).

In [ ]:
t3 = task3(client, questions)
for label, p in t3["registered"].items():
    print(f"{label}: registry version {p.version} | commit: {p.commit_message} | "
          f"tags: {p.tags}")

In [ ]:
for name, diff in t3["diffs"].items():
    print(f"===== diff {name} =====")
    print(diff, "\n")

In [ ]:
t = t3["table"]
comparison = pd.DataFrame({
    "Prompt Version": t["prompt_version"],
    "Avg Latency (s)": t["avg_latency_seconds"].round(3),
    "Avg Input Tokens": t["avg_input_tokens"].round(1),
    "Avg Output Tokens": t["avg_output_tokens"].round(1),
    f"Avg Cost ({settings.currency})": t["avg_cost_per_request"].map(lambda v: f"{v:.8f}"),
    "Valid JSON Rate": t["valid_json"].round(2),
    "Schema Valid Rate": t["schema_valid_rate"].round(2),
    "Quality (keyword coverage)": t["quality_keyword_coverage"].round(2),
    f"Monthly cost 10k ({settings.currency})": t["projected_monthly_cost_10k"].round(4),
    "Recommended?": t["recommended"],
})
comparison

In [ ]:
prod = mlflow.genai.load_prompt(f"prompts:/{config.PROMPT_NAME}@{config.PRODUCTION_ALIAS}")
print(f"Alias '{config.PRODUCTION_ALIAS}' -> {config.PROMPT_NAME} version {prod.version}")
print(prod.template)

In [ ]:
d = t3["detail"]
for qid in ["Q1"]:
    for _, row in d[d["question_id"] == qid].iterrows():
        print(f"----- {qid} {row['prompt_version']} | valid_json={row['valid_json']} "
              f"| tokens out={row['output_tokens']} -----")
        print(row["response_text"][:700], "\n")

In [ ]:
b = t3["best"]
others = t[t["prompt_version"] != b["prompt_version"]]
print("DRAFT facts for the promotion justification (rewrite in your own words, >= 150 words):")
print(f"- Promoted: {b['prompt_version']} (registry version {b['registry_version']}).")
print(f"- valid_json {b['valid_json']:.2f}, schema valid {b['schema_valid_rate']:.2f}; "
      f"others: " + ", ".join(f"{r.prompt_version}={r.valid_json:.2f}" for r in others.itertuples()))
print(f"- quality (keyword coverage) {b['quality_keyword_coverage']:.2f}; others: "
      + ", ".join(f"{r.prompt_version}={r.quality_keyword_coverage:.2f}" for r in others.itertuples()))
print(f"- avg latency {b['avg_latency_seconds']:.3f}s; avg tokens in/out "
      f"{b['avg_input_tokens']:.0f}/{b['avg_output_tokens']:.0f}")
print(f"- avg cost {b['avg_cost_per_request']:.8f} {settings.currency}; monthly (10k) "
      f"{b['projected_monthly_cost_10k']:.4f} {settings.currency}")

### Promotion justification

**

In [ ]:
meta = write_metadata(client)
print(json.dumps(meta, indent=2))
print("\nFiles in outputs/:")
for p in sorted(config.OUTPUTS_DIR.iterdir()):
    print("  ", p.name)

## Conclusion